In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce

In [3]:
panel_clean = pd.read_csv("data/processed/ea_country_panel_clean.csv")


In [4]:
DATA_PROCESSED = Path("data/processed")

df = pd.read_csv(DATA_PROCESSED / "ea_country_panel_clean.csv")

# Ensure sorted
df = df.sort_values(["country", "year"]).reset_index(drop=True)

# 1) Extra lags for targets and macro vars
lag_cols = [
    "birth_rate",
    "mean_age_first_marriage",
    "hpi_growth",
    "hicp_growth",
    "gdp_growth",
    "unemp_rate"
]

for col in lag_cols:
    df[f"{col}_lag1"] = df.groupby("country")[col].shift(1)
    df[f"{col}_lag2"] = df.groupby("country")[col].shift(2)

# 2) Changes (first differences) for main outcomes
df["birth_rate_change"] = df["birth_rate"] - df["birth_rate_lag1"]
df["marriage_age_change"] = (
    df["mean_age_first_marriage"] - df["mean_age_first_marriage_lag1"]
)

# 3) Rolling 3-year means for macro vars
roll_cols = ["hpi_growth", "hicp_growth", "gdp_growth", "unemp_rate"]

for col in roll_cols:
    df[f"{col}_roll3_mean"] = (
        df.groupby("country")[col]
          .rolling(window=3, min_periods=2)
          .mean()
          .reset_index(level=0, drop=True)
    )

# 4) Drop rows where key lags are missing
core_cols = [
    "birth_rate", "mean_age_first_marriage",
    "hpi_growth_lag1", "hicp_growth_lag1",
    "gdp_growth_lag1", "unemp_rate_lag1",
    "birth_rate_lag1", "mean_age_first_marriage_lag1"
]

ml_df = df.dropna(subset=core_cols).reset_index(drop=True)

print("ML dataset shape:", ml_df.shape)
ml_df.head()


ML dataset shape: (1531, 28)


,country,year,birth_rate,mean_age_first_marriage,hpi_index,hicp_index,gdp_growth,unemp_rate,hpi_growth,hicp_growth,...,hpi_growth_lag2,hicp_growth_lag2,gdp_growth_lag2,unemp_rate_lag2,birth_rate_change,marriage_age_change,hpi_growth_roll3_mean,hicp_growth_roll3_mean,gdp_growth_roll3_mean,unemp_rate_roll3_mean
0,Austria,2014,9.6,32.25,94.68,98.42,0.8,5.633333,0.0,-0.985915,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,-0.382048,0.8,5.633333
1,Austria,2014,9.6,32.25,94.68,97.85,0.8,5.633333,0.0,-0.579151,...,0.0,0.221819,0.8,5.633333,0.0,0.0,0.0,-0.447749,0.8,5.633333
2,Austria,2014,9.6,32.25,94.68,1.50,0.8,5.633333,0.0,-98.467041,...,0.0,-0.985915,0.8,5.633333,0.0,0.0,0.0,-33.344036,0.8,5.633333
3,Austria,2014,9.6,32.25,94.68,1.80,0.8,5.633333,0.0,20.000000,...,0.0,-0.579151,0.8,5.633333,0.0,0.0,0.0,-26.348731,0.8,5.633333
4,Austria,2014,9.6,32.25,94.68,2.00,0.8,5.633333,0.0,11.111111,...,0.0,-98.467041,0.8,5.633333,0.0,0.0,0.0,-22.451977,0.8,5.633333


In [5]:
# Minimal version (only some columns) – good for a clean ML notebook
keep_cols = [
    "country", "year",
    "birth_rate", "birth_rate_change",
    "mean_age_first_marriage", "marriage_age_change",
    "hpi_index", "hicp_index",
    "hpi_growth", "hicp_growth",
    "gdp_growth", "unemp_rate",
    "hpi_growth_lag1", "hicp_growth_lag1",
    "gdp_growth_lag1", "unemp_rate_lag1",
    "birth_rate_lag1", "birth_rate_lag2",
    "mean_age_first_marriage_lag1", "mean_age_first_marriage_lag2",
]

# Also add rolling features if you want them in ML
keep_cols += [c for c in ml_df.columns if "roll3_mean" in c]

ml_for_models = ml_df[keep_cols].copy()

ml_for_models.to_csv(DATA_PROCESSED / "ea_ml_panel.csv", index=False)
ml_for_models.to_parquet(DATA_PROCESSED / "ea_ml_panel.parquet", index=False)

ml_for_models.head()


,country,year,birth_rate,birth_rate_change,mean_age_first_marriage,marriage_age_change,hpi_index,hicp_index,hpi_growth,hicp_growth,...,gdp_growth_lag1,unemp_rate_lag1,birth_rate_lag1,birth_rate_lag2,mean_age_first_marriage_lag1,mean_age_first_marriage_lag2,hpi_growth_roll3_mean,hicp_growth_roll3_mean,gdp_growth_roll3_mean,unemp_rate_roll3_mean
0,Austria,2014,9.6,0.0,32.25,0.0,94.68,98.42,0.0,-0.985915,...,0.8,5.633333,9.6,NaN,32.25,NaN,0.0,-0.382048,0.8,5.633333
1,Austria,2014,9.6,0.0,32.25,0.0,94.68,97.85,0.0,-0.579151,...,0.8,5.633333,9.6,9.6,32.25,32.25,0.0,-0.447749,0.8,5.633333
2,Austria,2014,9.6,0.0,32.25,0.0,94.68,1.50,0.0,-98.467041,...,0.8,5.633333,9.6,9.6,32.25,32.25,0.0,-33.344036,0.8,5.633333
3,Austria,2014,9.6,0.0,32.25,0.0,94.68,1.80,0.0,20.000000,...,0.8,5.633333,9.6,9.6,32.25,32.25,0.0,-26.348731,0.8,5.633333
4,Austria,2014,9.6,0.0,32.25,0.0,94.68,2.00,0.0,11.111111,...,0.8,5.633333,9.6,9.6,32.25,32.25,0.0,-22.451977,0.8,5.633333
